<a href="https://colab.research.google.com/github/Wezz-git/AI-samples/blob/main/MLOps_Model_Persistence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**The focus: **Production Readiness.

**Project:** Saving the Churn Model Pipeline.

**Goal:** To correctly serialize (save) a trained model and its crucial preprocessing steps (StandardScaler, OneHotEncoder) to disk using the pickle library. This allows a deployed app to load the model instantly, making the app launch fast, efficient, and reliable.

Retrain and Save model

In [9]:
import pandas as pd
import numpy as np
import pickle
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load and preprocess data - (static public URL)
data_url = 'https://raw.githubusercontent.com/Wezz-git/AI-samples/refs/heads/main/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data_url)

# Clean and initialize preprocess
df.drop('customerID', axis=1, inplace=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df.replace({'Yes' : 1, 'No' : 0}, inplace=True)

# One-hot encoding
df_processed = pd.get_dummies(df, drop_first=True)

# Split data into training and testing sets
# Target is Churn column
X = df_processed.drop(columns='Churn')    # X - Drop single 'Churn' Column
y = df_processed['Churn']

# Split Data (X_train for model)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model_xgb.fit(X_train, y_train)

# Save trained model as .pkl file
model_filename = 'Churn_model.pkl'


with open(model_filename, 'wb') as file:   #'wb' = 'Write Binary'
    pickle.dump(model_xgb, file)

print(f"Model saved successfully to {model_filename}")

/tmp/ipython-input-1851425173.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({'Yes' : 1, 'No' : 0}, inplace=True)


Model saved successfully to Churn_model.pkl


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [12:46:16] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation (load model)

In [10]:
# Load trained model from .pkl file
loaded_model = pickle.load(open('Churn_model.pkl', 'rb'))     # 'rb' = Read Binary

# Make prediction with the model (Validation)
# Quick check using first 5 rows
sample_data = X_test.head(5)
sample_predictions = loaded_model.predict(sample_data)

print("Sample Predictions:")
print(f"Predictions from loaded model : {sample_predictions}")
#

Sample Predictions:
Predictions from loaded model : [1 0 0 1 0]
